# 01 — Tratamento e Carga de Dados (SAML-D)

Este notebook documenta a etapa oficial de ingestão, tipagem compacta, limpeza e preparação do
dataset sintético **SAML-D** no **FraudRadar PLD**.

**Princípios arquiteturais:**
- Toda a lógica reutilizável vive no pacote compartilhado `src/fraudradar/data/` (`schema.py` e
  `saml_d.py`) — o notebook apenas importa e demonstra o fluxo.
- **Separação anti-vazamento:** as colunas de gabarito (`Is_laundering` e `Laundering_type`) são
  separadas na carga em uma tabela própria (`ground_truth`), e nunca entram nas features nem na
  inferência do modelo em produção.
- **Uso de memória controlado:** com `CSV_DTYPES`, os ~9,5 milhões de registros ocupam menos de
  450 MB em RAM em vez de vários gigabytes.


In [1]:
import tempfile
from pathlib import Path

import pandas as pd

from fraudradar.config import ROOT_DIR
from fraudradar.data.saml_d import TRANSACTION_COLUMNS, clean_typology, load_csv
from fraudradar.data.schema import (
    AMOUNT_CURRENCY,
    COLUMN_RENAMES,
    CSV_DTYPES,
    GROUND_TRUTH_COLUMNS,
)

CSV_PATH = ROOT_DIR / "data" / "SAML-D.csv"
print(f"Arquivo de entrada: {CSV_PATH.name} ({CSV_PATH.stat().st_size / 1024**2:.1f} MB)")
print(f"Moeda padronizada dos valores (Amount): {AMOUNT_CURRENCY}")
print(f"Colunas de gabarito isoladas: {GROUND_TRUTH_COLUMNS}")

Arquivo de entrada: SAML-D.csv (950.0 MB)
Moeda padronizada dos valores (Amount): GBP
Colunas de gabarito isoladas: ['Is_laundering', 'Laundering_type']


## 1. Esquema e dtypes compactos (`fraudradar.data.schema`)

Antes de carregar o CSV completo, definimos tipos compactos:
- contas (`Sender_account`, `Receiver_account`) têm até 10 dígitos e usam `int64`;
- moedas, países, meios de pagamento e tipologias viram `category`;
- `Is_laundering` usa `int8`;
- `Date` e `Time` são lidos como `string` e combinados em `occurred_at` (`datetime64`).

A coluna `Amount` é interpretada em libras esterlinas (`GBP`), independentemente de
`Payment_currency`, pois as medianas são equivalentes (~6,5 mil a 7,7 mil) entre todas as 13 moedas
do SAML-D.


In [2]:
schema_df = pd.DataFrame(
    [
        {
            "coluna_csv": col,
            "coluna_banco": COLUMN_RENAMES.get(col, "occurred_at (Date + Time)"),
            "dtype_leitura": CSV_DTYPES[col],
            "destino": "ground_truth" if col in GROUND_TRUTH_COLUMNS else "transactions",
        }
        for col in CSV_DTYPES
    ]
)
schema_df

,coluna_csv,coluna_banco,dtype_leitura,destino
0,Time,occurred_at (Date + Time),string,transactions
1,Date,occurred_at (Date + Time),string,transactions
2,Sender_account,sender,int64,transactions
3,Receiver_account,receiver,int64,transactions
4,Amount,amount,float64,transactions
5,Payment_currency,payment_currency,category,transactions
6,Received_currency,received_currency,category,transactions
7,Sender_bank_location,sender_country,category,transactions
8,Receiver_bank_location,receiver_country,category,transactions
9,Payment_type,payment_type,category,transactions


## 2. Carga e separação entre `transactions` e `ground_truth` (`load_csv`)

A função `fraudradar.data.saml_d.load_csv` lê o arquivo, converte `Date + Time` em `occurred_at`,
atribui `id` sequencial e devolve dois `DataFrame`s separados:
1. **`transactions`**: apenas as colunas operacionais visíveis ao banco e ao modelo;
2. **`ground_truth`**: o rótulo (`label`) e a tipologia (`original_typology` e `typology`), usados
   exclusivamente em treino controlado e avaliação offline.


In [3]:
tx, gt = load_csv(CSV_PATH)

tx_mb = tx.memory_usage(deep=True).sum() / 1024**2
gt_mb = gt.memory_usage(deep=True).sum() / 1024**2
print(f"transactions: {len(tx):,} linhas x {tx.shape[1]} colunas ({tx_mb:.1f} MB)")
print(f"ground_truth: {len(gt):,} linhas x {gt.shape[1]} colunas ({gt_mb:.1f} MB)")
print(f"Período coberto: {tx['occurred_at'].min()} até {tx['occurred_at'].max()}")
display(tx.head(5))
display(gt.head(5))

transactions: 9,504,852 linhas x 10 colunas (371.6 MB)
ground_truth: 9,504,852 linhas x 4 colunas (63.5 MB)
Período coberto: 2022-10-07 10:35:19 até 2023-08-23 10:57:12


,id,occurred_at,sender,receiver,amount,payment_currency,received_currency,sender_country,receiver_country,payment_type
0,0,2022-10-07 10:35:19,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit
1,1,2022-10-07 10:35:20,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border
2,2,2022-10-07 10:35:20,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque
3,3,2022-10-07 10:35:21,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH
4,4,2022-10-07 10:35:21,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit


,transaction_id,label,original_typology,typology
0,0,0,Normal_Cash_Deposits,normal_cash_deposits
1,1,0,Normal_Fan_Out,normal_fan_out
2,2,0,Normal_Small_Fan_Out,normal_small_fan_out
3,3,0,Normal_Fan_In,normal_fan_in
4,4,0,Normal_Cash_Deposits,normal_cash_deposits


## 3. Verificações de qualidade dos dados

Conferimos valores ausentes, faixa de valores monetários, contas distintas e colunas de saída.


In [4]:
quality = pd.DataFrame(
    {
        "nulos": tx.isna().sum(),
        "dtype": tx.dtypes.astype(str),
        "valores_unicos": [tx[c].nunique() for c in tx.columns],
    }
)
display(quality)

print(f"Valores negativos em amount: {(tx['amount'] < 0).sum()}")
print(f"Valores iguais a zero em amount: {(tx['amount'] == 0).sum()}")
print(f"Contas remetentes distintas: {tx['sender'].nunique():,}")
print(f"Contas destinatárias distintas: {tx['receiver'].nunique():,}")
expected_cols = ["id", "occurred_at", *TRANSACTION_COLUMNS[2:]]
print("Colunas de transactions batem com o contrato:", list(tx.columns) == expected_cols)

,nulos,dtype,valores_unicos
id,0,int32,9504852
occurred_at,0,datetime64[us],7753911
sender,0,int64,292715
receiver,0,int64,652266
amount,0,float64,2314277
payment_currency,0,category,13
received_currency,0,category,13
sender_country,0,category,18
receiver_country,0,category,18
payment_type,0,category,7


Valores negativos em amount: 0
Valores iguais a zero em amount: 0
Contas remetentes distintas: 292,715


Contas destinatárias distintas: 652,266
Colunas de transactions batem com o contrato: True


## 4. Normalização das tipologias (`clean_typology`)

Os nomes brutos de `Laundering_type` vêm com espaços, hifens, maiúsculas e um erro de grafia do
dataset original (`Normal_Foward`). A função `clean_typology` padroniza tudo em `snake_case`
(ex.: `Stacked Bipartite` → `stacked_bipartite` e `Normal_Foward` → `normal_forward`).


In [5]:
typology_map = (
    gt[["original_typology", "typology", "label"]]
    .drop_duplicates()
    .sort_values(["label", "typology"], ascending=[False, True])
    .reset_index(drop=True)
)
print(
    f"Total de tipologias distintas: {len(typology_map)} "
    f"({(typology_map['label'] == 1).sum()} suspeitas e "
    f"{(typology_map['label'] == 0).sum()} normais)"
)
print("Exemplo da correção de grafia:", clean_typology("Normal_Foward"))
display(typology_map)

Total de tipologias distintas: 28 (17 suspeitas e 11 normais)
Exemplo da correção de grafia: normal_forward


,original_typology,typology,label
0,Behavioural_Change_1,behavioural_change_1,1
1,Behavioural_Change_2,behavioural_change_2,1
2,Bipartite,bipartite,1
3,Cash_Withdrawal,cash_withdrawal,1
4,Cycle,cycle,1
5,Deposit-Send,deposit_send,1
6,Fan_In,fan_in,1
7,Fan_Out,fan_out,1
8,Gather-Scatter,gather_scatter,1
9,Layered_Fan_In,layered_fan_in,1


## 5. Persistência colunar em Parquet

O formato Parquet preserva os dtypes (inclusive `category` e `datetime64`) com forte compressão
colunar. Abaixo demonstramos a escrita e releitura em Parquet em um diretório temporário (para não
duplicar arquivos grandes no disco do projeto).


In [6]:
sample_tx = tx.head(250_000)
with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = Path(tmpdir) / "saml_d_sample.parquet"
    sample_tx.to_parquet(parquet_path, index=False)
    reloaded = pd.read_parquet(parquet_path)
    size_mb = parquet_path.stat().st_size / 1024**2
    same_cols = list(reloaded.columns) == list(sample_tx.columns)
    print(f"Parquet (250 mil linhas): {size_mb:.2f} MB | colunas preservadas: {same_cols}")
    print(f"Dtypes preservados: {reloaded.dtypes.equals(sample_tx.dtypes)}")

Parquet (250 mil linhas): 6.63 MB | colunas preservadas: True
Dtypes preservados: True


## 6. Carga no PostgreSQL (`worker load`)

Em produção e nos testes locais com banco, o worker (`worker/load.py`) faz a carga em streaming por
`COPY` diretamente para o PostgreSQL, mantendo a separação entre `transactions`, `ground_truth` e
a tabela derivada `accounts`, além de atualizar os metadados do conjunto em `public.datasets`
(`labeled_weeks` e `has_laundering`).

Comando oficial (não executado aqui para não alterar o banco local):
```bash
uv run python -m worker load
```
